In [21]:
%load_ext autoreload
%autoreload 2


In [17]:
import pandas as pd
import os
import warnings
# Suppress SSL and general warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", message="Unverified HTTPS request")

In [12]:
companies_df = pd.read_csv("./private_data/companies.csv")[["name", "location", "domain", "id", "tagline"]]
enriched_df = pd.read_csv("./private_data/enriched_companies.csv")
for index, row in enriched_df.iterrows():
    location = row["Location"]
    if pd.isna(location):
        id = row["id"]
        company = companies_df[companies_df["id"] == id]
        location = company["location"].values[0]
        enriched_df.at[index, "Location"] = location

enriched_df.to_csv("./private_data/enriched_companies.csv", index=False)

In [15]:
funding_rounds = pd.read_csv("./private_data/funding_rounds.csv")
funding_ids = sorted(funding_rounds["company_id"].unique().tolist())
enriched_ids = sorted(enriched_df["id"].unique().tolist())

In [11]:
import imaplib
import email
from email.header import decode_header
from bs4 import BeautifulSoup
from config import EMAIL_ADDRESS, EMAIL_PASSWORD, IMAP_SERVER, WATCHLIST
from uid_tracker import is_uid_seen, mark_uid_seen
from parser import ctvc, ctvc_date 

def fetch_unread_newsletters():
    imap = imaplib.IMAP4_SSL(IMAP_SERVER)
    imap.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
    imap.select("Inbox")

    messages = []

    for sender in WATCHLIST:
        sender = sender.strip()
    
        status, response = imap.search(None, f'FROM "{sender}"')
        if status != "OK":
            print(f"❌ Failed to search inbox for {sender}")
            continue

        email_ids = response[0].split()
        for eid in reversed(email_ids):  # latest first
            uid = eid.decode()
            # if is_uid_seen(sender, uid):
            #     continue

            status, msg_data = imap.fetch(eid, "(RFC822)")
            if status != "OK":
                continue
            raw_email = msg_data[0][1]
            msg = email.message_from_bytes(raw_email)
            html_content = None
            if msg.is_multipart():
                for part in msg.walk():
                    if part.get_content_type() == "text/html":
                        html_content = part.get_payload(decode=True).decode("utf-8", errors="ignore")
                        break
            else:
                html_content = msg.get_payload(decode=True).decode("utf-8", errors="ignore")

            if html_content:
                messages.append({
                    "sender": sender,
                    "uid": uid,
                    "html": html_content,
                    "subject": msg["Subject"]
                })
                # mark_uid_seen(sender, uid)

    imap.logout()
    return messages


import imaplib
import email
from email.header import decode_header
from bs4 import BeautifulSoup
from config import EMAIL_ADDRESS, EMAIL_PASSWORD, IMAP_SERVER, WATCHLIST
from uid_tracker import is_uid_seen, mark_uid_seen


def fetch_unread_newsletters():
    imap = imaplib.IMAP4_SSL(IMAP_SERVER)
    imap.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
    imap.select("Inbox")

    messages = []

    for sender in WATCHLIST:
        sender = sender.strip()
        status, response = imap.search(None, f'FROM "{sender}"')
        if status != "OK":
            print(f"❌ Failed to search inbox for {sender}")
            continue

        email_ids = response[0].split()
        for eid in reversed(email_ids):  # latest first
            uid = eid.decode()
            # if is_uid_seen(sender, uid):
            #     continue

            status, msg_data = imap.fetch(eid, "(RFC822)")
            if status != "OK":
                continue

            raw_email = msg_data[0][1]
            msg = email.message_from_bytes(raw_email)

            html_content = None
            text_content = None
            if msg.is_multipart():
                for part in msg.walk():
                    content_type = part.get_content_type()
                    # print(f"THIS IS THE CONTENT TYPE {content_type}")
                    if content_type == "text/html":
                        html_content = part.get_payload(decode=True).decode("utf-8", errors="ignore")
                    elif content_type == "text/plain":
                        text_content = part.get_payload(decode=True).decode("utf-8", errors="ignore")
            else:
                content_type = msg.get_content_type()
                # print(f"THIS IS THE CONTENT TYPE {content_type}")

                if content_type == "text/html":
                    html_content = msg.get_payload(decode=True).decode("utf-8", errors="ignore")
                elif content_type == "text/plain":
                    text_content = msg.get_payload(decode=True).decode("utf-8", errors="ignore")

            if html_content or text_content:
                messages.append({
                    "sender": sender,
                    "uid": uid,
                    "html": html_content,
                    "text": text_content,
                    "subject": msg["Subject"]
                })
                # mark_uid_seen(sender, uid)

    imap.logout()
    return messages

print(f"{WATCHLIST=}")
result = fetch_unread_newsletters()
sender = result[0]["sender"]
# sender_function_map = {"hello@ctvc.co",:None,"newsletter@keepcool.co":None}
html = (result[0]["html"])

# ctvc(html)

WATCHLIST=['hello@ctvc.co', 'newsletter@keepcool.co', 'climatetech@dealflowweekly.com']


In [16]:
from datetime import datetime
def ctvc_date(html: str) -> datetime.date:
    """
    Extracts the publication date from a CTVC newsletter URL and returns it as a date object (YYYY-MM-DD).
    """
    try:
        # response = requests.get(url, verify=False, timeout=5)
        # response.raise_for_status()
        soup = BeautifulSoup(html, "html.parser")
        date_element = soup.find('span', class_='post-meta-date')

        if date_element:
            date_text = date_element.get_text(strip=True)
        return datetime.strptime(date_text, "%d %b %Y")

        # date_tag = soup.find("time")
        # if date_tag and 'datetime' in date_tag.attrs:
        #     date_str = date_tag['datetime']
        #     return datetime.fromisoformat(date_str.replace("Z", "+00:00")).date()
        # else:
        #     raise ValueError("No <time> tag with datetime attribute found.")
    except Exception as e:
        raise RuntimeError(f"Failed to fetch or parse date: {e}")
    

html

'<!doctype html>\r\n<html>\r\n    <head>\r\n        <meta name="viewport" content="width=device-width">\r\n        <meta http-equiv="Content-Type" content="text/html; charset=UTF-8">\r\n        <!--[if mso]><xml><o:OfficeDocumentSettings><o:PixelsPerInch>96</o:PixelsPerInch><o:AllowPNG/></o:OfficeDocumentSettings></xml><![endif]-->\r\n        <title>&#x1F30E; Taxing times for CCUS #254</title>\r\n        <style>\r\n.post-title-link {\r\n  display: block;\r\n  margin-top: 32px;\r\n  color: #15212A;\r\n  text-align: center;\r\n  line-height: 1.1em;\r\n}\r\n.post-title-link-left {\r\n  text-align: left;\r\n}\r\n.view-online-link {\r\n  word-wrap: none;\r\n  white-space: nowrap;\r\n  color: #15212a;\r\n  color: rgba(0, 0, 0, 0.6);\r\n  text-decoration: underline !important;\r\n}\r\n.kg-nft-link {\r\n  display: block;\r\n  text-decoration: none !important;\r\n  color: #15212A !important;\r\n  font-family: inherit !important;\r\n  font-size: 14px;\r\n  line-height: 1.3em;\r\n  padding-top: 4

In [38]:
import pandas as pd

companies = pd.read_csv("private_data/companies.csv")[["Company","Domain","Location","Tagline","Investors"]]
enrich_companies = pd.read_csv("private_data/enrich_companies.csv")
for index, row in enrich_companies.iterrows():
    enrich_location = row["Location"]
    enrich_investors = row["Investors"]
    if pd.isna(enrich_location):
        enrich_companies.at[index,"Location"] = companies.loc[index,"Location"]
    if pd.isna(enrich_investors):
        enrich_companies.at[index,"Investors"] = companies.loc[index,"Investors"]


In [39]:
enrich_companies.to_csv("private_data/enrich_companies.csv")

In [58]:
from affinity import affinity_enrich
# Count companies not in Affinity
not_found_companies = enrich_companies[enrich_companies["Affinty ID"] == "Not in Affinity"].shape[0]

# Iterate through enrich_companies
count = 1
for index, row in enrich_companies.iterrows():
    if row["Affinty ID"] == "Not in Affinity":
        # if count%5==0:
        print(f"Processing {count}/{not_found_companies}")
        info_for_enrich = (
            row["name"],
            row["Location"],
            row["domain"],
            row["id"],
            row["tagline"],
            row["Investors"]
        )
        
        # Enrich
        result = affinity_enrich(info_for_enrich, 0)

        # Update row in DataFrame
        for key, value in result.items():
            if key in enrich_companies.columns:
                enrich_companies.at[index, key] = value
        count+=1
        # break  # remove this if you want to run on ALL not-in-Affinity entries


Processing 1/1468
Processing 2/1468
Processing 3/1468
Processing 4/1468
Processing 5/1468
Processing 6/1468
Processing 7/1468
Processing 8/1468
Processing 9/1468
Processing 10/1468
Processing 11/1468
Processing 12/1468
Processing 13/1468
Processing 14/1468
Processing 15/1468
Processing 16/1468
Processing 17/1468
Processing 18/1468
Processing 19/1468
Processing 20/1468
Processing 21/1468
Processing 22/1468
Processing 23/1468
Processing 24/1468
Processing 25/1468
Processing 26/1468
Processing 27/1468
Processing 28/1468
Processing 29/1468
Processing 30/1468
Processing 31/1468
Processing 32/1468
Processing 33/1468
Processing 34/1468
Processing 35/1468
Processing 36/1468
Processing 37/1468
Processing 38/1468
Processing 39/1468
Processing 40/1468
Processing 41/1468
Processing 42/1468
Processing 43/1468
Processing 44/1468
Processing 45/1468
Processing 46/1468
Processing 47/1468
Processing 48/1468
Processing 49/1468
Processing 50/1468
Processing 51/1468
Processing 52/1468
Processing 53/1468
Pr

In [8]:
import pandas as pd
import os 

enrich_companies = pd.read_csv("private_data/enriched_companies.csv")

In [9]:
enrich_companies[enrich_companies["Affinty ID"] == "Not in Affinity"]

,Unnamed: 0,id,name,tagline,domain,Location,Investors,Employees (Current),Employees: Growth YoY (%),Investment Stage,Last Funding Amount (USD),LinkedIn Profile (Founders/CEOs),In Energize Affinity,Affinty ID
1,1,ca38d2f4-157d-4146-a67e-9df8d5e3d72b,11x,AI autonomous digital workers developer,https://www.11x.com/,"San Francisco, CA","[' sv angel', ' quiet capital', 'benchmark', '...",NaN,NaN,NaN,NaN,NaN,False,Not in Affinity
4,4,a5dcbe42-0f73-4697-b9bf-ad1b08e1637b,1mt nation,Developer of nature-based carbon removal projects,NaN,"Tallinn, Estonia","[' warmeston', 'margus kohava']",NaN,NaN,NaN,NaN,NaN,False,Not in Affinity
10,10,38af1a43-6956-4c88-a073-fe52657babda,257,AI-powered home energy management software,NaN,"New York, New York",['f2 venture capital'],NaN,NaN,NaN,NaN,NaN,False,Not in Affinity
16,16,2221eb02-6e36-4275-9bce-f8cd4bb78846,44.01,Carbon removal and mineralization technologies,NaN,"London, U.K.","['nysnø climate investments', 'equinor venture...",NaN,NaN,NaN,NaN,NaN,False,Not in Affinity
17,17,74c55a8d-682f-4ee9-9755-1175081724aa,4elements,Launch of new high-impact climate startups,NaN,"TK, France",[],NaN,NaN,NaN,NaN,NaN,False,Not in Affinity
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4513,4513,5a25b043-478a-445a-8555-8d0a6ffc4b3c,zikooin,Provider of meat alternatives,NaN,South Korea,['mostly us venture firms'],NaN,NaN,NaN,NaN,NaN,False,Not in Affinity
4519,4519,409ead0f-640b-4954-8b1b-8badf3de00e6,zoomo,Maker of electric last-mile delivery vehicles,NaN,Australia,"[' contrarian ventures', 'collaborative fund',...",NaN,NaN,NaN,NaN,NaN,False,Not in Affinity
4520,4520,aeb715e9-0491-4002-b470-f5ecce542747,zordi,Autonomous greenhouse developer,NaN,"Boston, MA",['khosla ventures'],NaN,NaN,NaN,NaN,NaN,False,Not in Affinity
4527,4527,955465a8-1e8f-49e2-aafe-8c75b0172742,zyngo ev,Sustainable last-mile solutions company,NaN,"Haryana, India","['delta corp holdings', ' lc nueva investment ...",NaN,NaN,NaN,NaN,NaN,False,Not in Affinity


In [40]:
from config import OPENAI_API_KEY, SUPABASE_API_KEY,SUPABASE_API_URL
from supabase import create_client, Client
from openai import OpenAI
import pandas as pd

client = OpenAI()

supabase: Client = create_client(SUPABASE_API_URL,SUPABASE_API_KEY)

def build_embedding_text(row):
    parts = [
        f"Company: {row['name']}",
        f"Tagline: {row['tagline'] or ''}",
        f"Domain: {row['domain'] or ''}",
        f"Location: {row['Location'] or ''}",
        f"Industry: {row['Industry'] or ''}",
        f"Investors: {row['Investors'] or ''}",
    ]
    return "\n".join([p for p in parts if p.strip()])

def fetch_all_companies_without_embeddings(batch_size=1000, max_rows=5000):
    all_rows = []
    for offset in range(0, max_rows, batch_size):
        response = supabase.table("companies")\
            .select("*")\
            .filter("embedding", "is", "null")\
            .range(offset, offset + batch_size - 1)\
            .execute()

        rows = response.data
        if not rows:
            break  # Exit early if no more results
        all_rows.extend(rows)

    return all_rows


def generate_embedding(text):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding

def update_embedding(company_id, embedding_vector):
    supabase.table("companies").update({
        "embedding": embedding_vector
    }).eq("id", company_id).execute()

def embed_batch():
    rows = fetch_all_companies_without_embeddings()
    for index, row in enumerate(rows):
        try:
            text = build_embedding_text(row)
            embedding = generate_embedding(text)
            update_embedding(row["id"], embedding)
            if index%20==0:
                print(f"processing {index}/{len(rows)}")
                print(f"✅ Embedded: {row['name']}")
        except Exception as e:
            print(f"❌ Error on {row['id']}: {e}")

df = pd.DataFrame(fetch_all_companies_without_embeddings())


In [41]:
df

,id,name,tagline,domain,Location,Investors,Employees (Current),Employees: Growth YoY (%),Investment Stage,Last Funding Amount (USD),LinkedIn Profile (Founders/CEOs),In Energize Affinity,Affinity ID,Industry,Business Models,Technologies,embedding
0,447ce3ec-66ef-42bb-b9e5-406c8673b6a1,Aether Fuels,Sustainable aviation fuel developer,aetherfuels.com,"Chicago, Illinois","['TechEnergy Ventures', 'Xora Innovation', 'Fo...",28.0,27.27,Series A,34000000.0,Conor Madigan (Founder and CEO): https://linke...,True,290082831,"['Transportation', 'Energy', 'Clean Energy']",Manufacturing,Hardware,None
1,69e7dc0a-f0ce-4bac-acca-f3a3c946c1ad,Aetherflux,Network of small satellites designed to collec...,aetherflux.com,"San Francisco, California","['others', 'Interlagos', 'Index Ventures', 'NE...",10.0,NaN,Series A,50000000.0,None,True,296564490,"['Space', 'Energy', 'Clean Energy']",Manufacturing,None,None
2,0921776b-48e1-49a7-ad2f-4976a656b430,Applied EV,Modular autonomous electric vehicle maker,aevrobotics.com,Australia,,NaN,NaN,None,NaN,None,False,291892991,None,None,None,None
3,453a406b-0163-4c94-98ab-fdb39bce8f10,Aeva,LiDAR sensing and perception technologies,aeva.com,"Mountain View, CA",['sylebra capital'],NaN,NaN,None,NaN,None,False,291723026,None,None,None,None
4,8f4b774d-0a70-4743-8685-f96947f284c4,afresh,AI company developing food technology that pre...,https://freshbus.in/,"San Francisco, CA","[' shell ventures', ' bright pixel', ' alteria...",NaN,NaN,None,NaN,None,False,Not in Affinity,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4385,e8a7e737-9de6-4d07-a81b-5862c52fdb40,ZwitterCo,Fouling-resistant membrane technologies provider,zwitterco.com,"Woburn, Massachusetts","['Genoa Ventures', 'Heritage Group Ventures', ...",66.0,-2.94,Series B,58400000.0,['Christopher Drover (CTO & Co-Founder): https...,True,198217648,"['Energy', 'Water', 'Waste Solution']",Manufacturing,None,None
4386,aaf88137-a2a4-4eba-975e-344f65abb453,Zymofix,Solid-state fermentation technology for fertil...,zymofix.com,"Ghent, Belgium",['high-tech gründerfonds'],NaN,NaN,None,NaN,None,False,299792881,None,None,None,None
4387,26398d63-642d-4fc1-88fd-81c9669707af,Zymvol Biomodeling,Biotech enzyme discovery platform,zymvol.com,"Barcelona, Spain","['Elaia Partners', 'Faber', 'Übermorgen Ventur...",35.0,-7.89,Seed,3000000.0,['Emanuele Monza (Co-founder & Chief Scientifi...,True,142375942,"['Health', 'Pharmaceutical', 'BioTechnology']",Manufacturing,"['Hardware', 'Deep Tech', 'Quantum Technologies']",None
4388,955465a8-1e8f-49e2-aafe-8c75b0172742,zyngo ev,Sustainable last-mile solutions company,None,"Haryana, India","['delta corp holdings', ' lc nueva investment ...",NaN,NaN,None,NaN,None,False,Not in Affinity,None,None,None,None


In [42]:
embed_batch()

processing 0/4390
✅ Embedded: Aether Fuels
processing 20/4390
✅ Embedded: AgriTask
processing 40/4390
✅ Embedded: Aigen
processing 60/4390
✅ Embedded: Aligned Climate Capital
processing 80/4390
✅ Embedded: algama
processing 100/4390
✅ Embedded: Alrik
processing 120/4390
✅ Embedded: aluna
processing 140/4390
✅ Embedded: Ampacimon
processing 160/4390
✅ Embedded: Anodot
processing 180/4390
✅ Embedded: Applied Carbon
processing 200/4390
✅ Embedded: Arbor
processing 220/4390
✅ Embedded: Arevo AB
processing 240/4390
✅ Embedded: Ascend Elements
processing 260/4390
✅ Embedded: Ati Motors
processing 280/4390
✅ Embedded: Augmentus
processing 300/4390
✅ Embedded: avant
processing 320/4390
✅ Embedded: bacta
processing 340/4390
✅ Embedded: BatX Energies
processing 360/4390
✅ Embedded: Bees & Bears
processing 380/4390
✅ Embedded: Bevi
processing 400/4390
✅ Embedded: Bioeutectics
processing 420/4390
✅ Embedded: Blackhorn Ventures
processing 440/4390
✅ Embedded: Blue Planet Energy Systems, LLC
process

In [86]:
query = "Solarcycle, Oakland, CA"
query_embedding = client.embeddings.create(
    model="text-embedding-3-small",
    input=query
).data[0].embedding

# Find a way to also process the querry into something that is more manageable
def search_companies_by_semantics(query_embedding, top_k=10):
    embedding_str = str(query_embedding)  # Converts list to PostgreSQL array syntax
    response = supabase.rpc(
        "match_companies_by_embedding",
        {"query_embedding": embedding_str, "match_count": top_k}
    ).execute()
    return response.data

for index, row in pd.DataFrame(search_companies_by_semantics(query_embedding)).iterrows():
    print(row.to_dict())
    break


{'name': 'SOLARCYCLE', 'tagline': 'Solar panel recycling company', 'domain': 'solarcycle.us', 'Location': 'Oakland, California'}


In [67]:
df[df["name"] =="tyba"]

,id,name,tagline,domain,Location,Investors,Employees (Current),Employees: Growth YoY (%),Investment Stage,Last Funding Amount (USD),LinkedIn Profile (Founders/CEOs),In Energize Affinity,Affinity ID,Industry,Business Models,Technologies,embedding


In [14]:
import time
import imaplib
import email
from bs4 import BeautifulSoup
from uid_tracker import is_uid_seen, mark_uid_seen
from db_client import insert_deals, in_database, fetch_all_companies
from config import EMAIL_ADDRESS, EMAIL_PASSWORD, IMAP_SERVER, WATCHLIST
from website_monitor import get_new_ctvc, get_new_forutne
from parser import ctvc, fortune, cleaning
from affinity import enrich_df
import pandas as pd
from ctvc import get_new_ctvc
from fortune import get_new_forutne

In [4]:

# import datawrangler

new_ctvc = get_new_ctvc()
new_fortune = get_new_forutne()
new_newsletters = new_ctvc + new_fortune
ctvc_df = pd.DataFrame()
print(f"We have {len(new_ctvc)} CTVC newsletters")
c_count = 0
for new_url in new_ctvc:
    print(c_count)
    c_count+=1
    df = ctvc(new_url)
    if df is not None and not df.empty:
        print("concatenating ctvc")
        ctvc_df = pd.concat([ctvc_df,df])
    else:
        print("failed")
    # break

# print(ctvc_df)
print(f"We have {len(new_fortune)} fortune newsletters")

fortune_df = pd.DataFrame()
f_count = 0
for new_url in new_fortune:
    print(f_count)
    f_count+=1
    df = fortune(new_url)
    if df is not None and not df.empty:
        print("concatonating fortune")
        fortune_df = pd.concat([fortune_df,df])
    else:
        print("failed")
    # break
    # if not fortune_df.empty:
    #     break

print("combining ctvc and fortune")
all_data = pd.concat([ctvc_df,fortune_df])



We have 5 CTVC newsletters
0
concatenating ctvc
1
concatenating ctvc
2
concatenating ctvc
3
concatenating ctvc
4
concatenating ctvc
We have 14 fortune newsletters
0
concatonating fortune
1
concatonating fortune
2
concatonating fortune
3
concatonating fortune
4
Retrying classify_industry...
Retrying a second time
Couldn't extract
failed
5
concatonating fortune
6
Retrying classify_industry...
concatonating fortune
7
Retrying classify_industry...
Retrying a second time
Couldn't extract
failed
8
Retrying classify_industry...
Retrying a second time
Couldn't extract
failed
9
Classification error: invalid syntax (<unknown>, line 1)
Retrying classify_industry...
concatonating fortune
10
concatonating fortune
11
Retrying classify_industry...
Retrying a second time
Couldn't extract
failed
12
concatonating fortune
13
Retrying classify_industry...
Retrying a second time
Couldn't extract
failed
combining ctvc and fortune


In [9]:
all_data.sort_values(by="name")

,name,deal_size,series,tagline,location,investors,domain,date,source
9,AMEA Power,$72M,Project Finance Debt,Renewable energy developer,"Dubai, United Arab Emirates",International Finance Corporation (IFC),https://ameapower.com/,2025-06-23,https://www.ctvc.co/epa-puts-corn-on-the-polic...
28,Abundia Global Impact Group,,,Waste into renewable fuels conversion service ...,"New York City, NY",,,2025-07-07,https://www.ctvc.co/bright-spots-and-sunsets-i...
5,Aedifion,$20M,Series B,Building energy optimization software developer,"Köln, Germany","Eurazeo, World Fund, BitStone Capital, Drees &...",https://www.aedifion.com/,2025-06-30,https://www.ctvc.co/overheard-at-lcaw-252/
23,Agrobiomics,$9M,Grant,Biostimulants for crop health developer,"Copenhagen, Denmark",European Innovation Council,https://www.agrobiomics.com/,2025-07-07,https://www.ctvc.co/bright-spots-and-sunsets-i...
1,Amogy,$23M,Growth,Clean fuel technology provider,"Brooklyn, NY","KDB Silicon Valley, Korea Development Bank, Bo...",https://www.amogy.co/,2025-07-21,https://www.ctvc.co/grwm-for-nycw-255/
...,...,...,...,...,...,...,...,...,...
0,Xelix,$160M,Series B,Developer of AI software in accounts payable,"New York City, NY","Insight Partners, Passion Capital, LocalGlobe",https://www.xelix.com/,2025-07-22,https://fortune.com/2025/07/22/jeff-dean-googl...
15,Zemetric,Undisclosed,,Transport electrification technology provider,"Silicon Valley, CA",,https://zemetric.com/,2025-07-21,https://www.ctvc.co/grwm-for-nycw-255/
6,captoplastic,$2M,Series A,Microplastics capture technology developer,"Vicálvaro, Spain",BeAble Capital,https://captoplastic.com/,2025-06-23,https://www.ctvc.co/epa-puts-corn-on-the-polic...
16,eMotion Fleet,$2M,Series A,Fleet electrification services provider,"Shinjuku-ku, Japan","Incubate Fund, Kyushu Open Innovation Fund No....",https://www.emotionfleet.com/,2025-07-07,https://www.ctvc.co/bright-spots-and-sunsets-i...


In [12]:
from parser import cleaning
print("cleaning")
companies, deals = cleaning(all_data)
print(f"final size of companies {companies.shape}")

companies

cleaning
Processing 1/112 rows...
Processing 21/112 rows...
Processing 41/112 rows...
Processing 61/112 rows...
Processing 81/112 rows...
Processing 101/112 rows...
final size of companies (112, 6)


,company,domain,location,tagline,investors,name
company_uuid,,,,,,
208e2efb-5bdc-4b27-9499-23ea0c55a61f,NaN,,"New York City, NY",Waste into renewable fuels conversion service ...,[],abundia global impact group
8a45bb4f-8ace-424a-bf8d-e28cad5bbc2f,NaN,https://www.aedifion.com/,"Köln, Germany",Building energy optimization software developer,"[eurazeo, hopp family office, world fund, dree...",aedifion
71eefbf9-c523-43e6-9b8d-bf3f6f746e50,NaN,https://www.agrobiomics.com/,"Copenhagen, Denmark",Biostimulants for crop health developer,[european innovation council],agrobiomics
ab168960-3869-457b-9ca3-b401fab975dc,NaN,https://ameapower.com/,"Dubai, United Arab Emirates",Renewable energy developer,[international finance corporation (ifc)],amea power
9032041b-bc9f-4115-8687-47aeb8c516e8,NaN,https://www.amogy.co/,"Brooklyn, NY",Clean fuel technology provider,"[jb investment, kdb silicon valley, bonangels ...",amogy
...,...,...,...,...,...,...
54f35751-0667-46ce-a671-87327edeeb63,NaN,https://www.vokbikes.com/,"Tallinn, Estonia",Heavy-duty electric cargo bike manufacturer,"[sunly, smartcap, specialist vc, sqm lithium v...",vok bikes
e4dd3999-8ee9-45e9-b7cc-77a0b5876097,NaN,https://www.woodchuck.com/,"Grand Rapids, MI",Streamlines wood waste and biomass processing ...,"[northstar clean energy, alloy partners, becke...",woodchuck
93f5c892-af1b-4941-8395-dd7bc3cdb6cd,NaN,https://xatoms.com/,"Toronto, Canada",AI-driven water purification materials developer,"[bdc capital, quantacet, boxone ventures, gene...",xatoms


In [13]:
from db_client import fetch_all

deals = pd.DataFrame(fetch_all(table="funding"))

In [19]:

def deal_in_database(row,deals,date_tolerance_days = 60):
    company_uuid = row["company_uuid"]
    if company_uuid in  deals["company_uuid"].values:
        date = datetime.strptime(row["date"],"%Y-%m-%d")
        matching_deals = deals[deals["company_uuid"] == company_uuid]
        matching_deals["date"] = pd.to_datetime(matching_deals["date"])
        for index, deal in matching_deals.iterrows():
            date_match = False
            if pd.notna(date) and pd.notna(deal["date"]):
                date_match = abs((date - deal["date"]).days)
                if date_match <= date_tolerance_days:
                    return True
    return False

row = {"company_uuid":"4ec18937-c30b-4034-9721-6c89bcef58cb","date":"2023-11-13"}

deal_in_database(row,deals)

True

In [9]:
from datetime import datetime
datetime.strptime("2022-03-14", "%Y-%m-%d")


datetime.datetime(2022, 3, 14, 0, 0)

In [ ]:
for index, row in enriched_companies.iterrows():
    from_db = in_database(row,db)
    if from_db is not False:
        existing_id = from_db["id"]
        temp_id = row["id"]
        to_change = deals[deals["company_id"] == temp_id]
        for i, deal in to_change.iterrows():
            deals.at[i,"company_id"] = existing_id
            # deals.at[]
    else:
        pass    # add row to database


In [3]:



import time
import imaplib
import email
from bs4 import BeautifulSoup
from uid_tracker import is_uid_seen, mark_uid_seen
from db_client import company_in_database, fetch_all, unseen_deals, upload_dataframe
from config import EMAIL_ADDRESS, EMAIL_PASSWORD, IMAP_SERVER, WATCHLIST
from parser import ctvc, fortune, cleaning
from affinity import enrich_df, affinity_enrich
import pandas as pd
from ctvc import get_new_ctvc
from fortune import get_new_forutne
import warnings
from embeddings import embed_companies
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", message="Unverified HTTPS request")

new_ctvc = get_new_ctvc()
new_fortune = get_new_forutne()
new_newsletters = new_ctvc + new_fortune
ctvc_df = pd.DataFrame()
print(f"We have {len(new_ctvc)} CTVC newsletters")
c_count = 0
for new_url in new_ctvc:
    print(c_count)
    c_count+=1
    df = ctvc(new_url)
    if df is not None and not df.empty:
        print("concatenating ctvc")
        ctvc_df = pd.concat([ctvc_df,df])
    else:
        print("failed")

print(f"We have {len(new_fortune)} fortune newsletters")



We have 5 CTVC newsletters
0
concatenating ctvc
1
concatenating ctvc
2
concatenating ctvc
3
concatenating ctvc
4
concatenating ctvc
We have 14 fortune newsletters


In [25]:

fortune_df = pd.DataFrame()
f_count = 0
for new_url in new_fortune:
    print(f_count)
    f_count+=1
    df = fortune(new_url)
    if df is not None and not df.empty:
        print("concatonating fortune")
        fortune_df = pd.concat([fortune_df,df])
    else:
        print("failed")
    break

print("combining ctvc and fortune")
all_data = pd.concat([ctvc_df,fortune_df])
companies, deals = cleaning(all_data)




0
concatonating fortune
combining ctvc and fortune
Processing 1/94 rows...
Processing 21/94 rows...
Processing 41/94 rows...
Processing 61/94 rows...
Processing 81/94 rows...


In [43]:
companies

,name,domain,location,tagline,investors
company_uuid,,,,,
ef66c221-2bf8-41e0-803c-1ba19f63b306,abundia global impact group,,"New York City, NY",Waste into renewable fuels conversion service ...,[houston american energy corp]
9b5963f9-3c27-41f9-b593-b481710d5dd4,aedifion,https://www.aedifion.com/,"Köln, Germany",Building energy optimization software developer,"[drees & sommer, hopp family office, world fun..."
543a0e42-a3aa-49fe-9da2-d5c162b166be,agrobiomics,https://agrobiomics.com/,"Copenhagen, Denmark",Biostimulants for crop health developer,[european innovation council]
3c299c69-bdbd-4a64-be3d-6acd1c7d6348,amea power,https://www.ameapower.com/,"Dubai, United Arab Emirates",Renewable energy developer,[international finance corporation (ifc)]
3c21ce93-9ec1-4375-b32c-cf455a598e20,amogy,https://www.amogy.co/,"Brooklyn, NY",Clean fuel technology provider,"[jb investment, bonangels venture partners, ko..."
...,...,...,...,...,...
c705bf1b-bd87-4613-93e4-d3529f1bba1a,tulum energy,https://tulumenergy.com/,"Luxembourg, Luxembourg",Clean hydrogen producer,"[mito technology (mito tech ventures), tdk ven..."
0b4ef153-728f-42a2-a048-39c98dd8cad7,ukrhydroenergo,,"Vyshgorodskiy Rayon, Ukraine",Hydropower generator,[european investment bank (eib)]
c9437a50-0958-4156-a694-8b4710760342,vok bikes,https://vokbikes.com/,"Tallinn, Estonia",Heavy-duty electric cargo bike manufacturer,"[sunly, metaplanet holdings, smartcap, special..."


In [6]:
from affinity import affinity_enrich
result = []
count = 0
for index, row in companies.iterrows():
    print(row["name"], count)
    result.append(affinity_enrich(row))
    count+=1
companies = pd.DataFrame(result)

abundia global impact group 0
aedifion 1
agrobiomics 2
amea power 3
amogy 4
antler bio 5
assetcool 6
atome 7
biomedit 8
blueredgold 9
busup 10
captoplastic 11
carbyon 12
cariqa 13
catalyxx 14
cellugy 15
chloris geospatial 16
clearway energy group 17
climate tech partners 18
climatiq 19
climeworks 20
coco robotics 21
colorifix 22
cosma 23
creative energy 24
crosstown h2r 25
currentt 26
dexter energy 27
doktar technologies 28
echandia 29
eeki foods 30
elemental advanced materials 31
emerald ai 32
emotion fleet 33
enter 34
eos energy enterprises 35
equitable earth 36
ess 37
evera cabs 38
fiber elements 39
fieldfactors 40
freight farms 41
ftc solar 42
geologicai 43
gridserve 44
halter 45
helical fusion 46
hived 47
hymeth 48
ignis h2 energy 49
insight m 50
intelligent energy 51
inyanga marine energy group 52
jälle technologies 53
lbc tank terminals 54
libre foods 55
light bridge 56
lightshift energy 57
loopworm 58
lumenstream 59
mars 60
membrion 61
minesto 62
momenta 63
move mobility 64
muf

In [44]:
companies

,name,domain,location,tagline,investors
company_uuid,,,,,
ef66c221-2bf8-41e0-803c-1ba19f63b306,abundia global impact group,,"New York City, NY",Waste into renewable fuels conversion service ...,[houston american energy corp]
9b5963f9-3c27-41f9-b593-b481710d5dd4,aedifion,https://www.aedifion.com/,"Köln, Germany",Building energy optimization software developer,"[drees & sommer, hopp family office, world fun..."
543a0e42-a3aa-49fe-9da2-d5c162b166be,agrobiomics,https://agrobiomics.com/,"Copenhagen, Denmark",Biostimulants for crop health developer,[european innovation council]
3c299c69-bdbd-4a64-be3d-6acd1c7d6348,amea power,https://www.ameapower.com/,"Dubai, United Arab Emirates",Renewable energy developer,[international finance corporation (ifc)]
3c21ce93-9ec1-4375-b32c-cf455a598e20,amogy,https://www.amogy.co/,"Brooklyn, NY",Clean fuel technology provider,"[jb investment, bonangels venture partners, ko..."
...,...,...,...,...,...
c705bf1b-bd87-4613-93e4-d3529f1bba1a,tulum energy,https://tulumenergy.com/,"Luxembourg, Luxembourg",Clean hydrogen producer,"[mito technology (mito tech ventures), tdk ven..."
0b4ef153-728f-42a2-a048-39c98dd8cad7,ukrhydroenergo,,"Vyshgorodskiy Rayon, Ukraine",Hydropower generator,[european investment bank (eib)]
c9437a50-0958-4156-a694-8b4710760342,vok bikes,https://vokbikes.com/,"Tallinn, Estonia",Heavy-duty electric cargo bike manufacturer,"[sunly, metaplanet holdings, smartcap, special..."


In [8]:
print(f" We have this many companies {companies.shape[0]}")
print(f" We have this many deals {deals.shape[0]}")

 We have this many companies 95
 We have this many deals 95


In [36]:
db = pd.DataFrame(fetch_all("companies"))
print("fetched all companies")



fetched all companies


In [37]:
db.columns

Index(['company_uuid', 'name', 'tagline', 'domain', 'location', 'investors',
       'Employees (Current)', 'Employees: Growth YoY (%)', 'Investment Stage',
       'Last Funding Amount (USD)', 'LinkedIn Profile (Founders/CEOs)',
       'In Energize Affinity', 'Affinity ID', 'Industry', 'Business Models',
       'Technologies', 'embedding'],
      dtype='object')

In [38]:
print(deals["company_uuid"].apply(type).value_counts())


company_uuid
<class 'str'>    94
Name: count, dtype: int64


In [40]:
import importlib, db_client
importlib.reload(db_client)

from db_client import (
    company_in_database,
    fetch_all,
    unseen_deals,
    upload_dataframe,
)


to_keep = pd.DataFrame()
count = 94
for index, row in companies.iterrows():
    print(f"We have {count} left")
    print(f"checking if --{row["name"].upper()}-- is in database")
    from_db = company_in_database(row,db)
    if from_db is not None:
        print(from_db.to_dict().keys())
        print(f"--{row["name"].upper()}-- is INDEED in the database")
        # print(f"from_db is type: {type(from_db)} and its value is {from_db} ")
        existing_id = from_db["company_uuid"]
        print("The existing id is", type(existing_id))
        print("The value of the existing is",existing_id)
        existing_name = from_db["name"]
        temp_id = row["company_uuid"]
        print("this is the temp_id :",temp_id, "And it is of type", type(temp_id))
        to_change = deals[deals["company_uuid"] == temp_id]
        for i, deal in to_change.iterrows():
            print(f"changed id from {temp_id} to {existing_id}")
            deals.at[i,"company_uuid"] = existing_id
            deals.at[i,"name"] = existing_name
    else:
        print(f"{row["name"]} is NOT in the database")
        to_keep = pd.concat([to_keep,row.to_frame().T])
    count-=1

deals = unseen_deals(deals)
companies = to_keep.apply(embed_companies,axis=1)

print(f"After checking the database we have {companies.shape[0]} new companies")

We have 94 left
checking if --ABUNDIA GLOBAL IMPACT GROUP-- is in database
abundia global impact group is NOT in the database
We have 93 left
checking if --AEDIFION-- is in database
aedifion is NOT in the database
We have 92 left
checking if --AGROBIOMICS-- is in database
agrobiomics is NOT in the database
We have 91 left
checking if --AMEA POWER-- is in database
amea power is NOT in the database
We have 90 left
checking if --AMOGY-- is in database
amogy is NOT in the database
We have 89 left
checking if --ANTLER BIO-- is in database
antler bio is NOT in the database
We have 88 left
checking if --ASSETCOOL-- is in database
assetcool is NOT in the database
We have 87 left
checking if --ATOME-- is in database
atome is NOT in the database
We have 86 left
checking if --BIOMEDIT-- is in database
biomedit is NOT in the database
We have 85 left
checking if --BLUEREDGOLD-- is in database
blueredgold is NOT in the database
We have 84 left
checking if --BUSUP-- is in database
busup is NOT in the

KeyboardInterrupt: 

In [ ]:
deals

In [ ]:
companies

In [ ]:
upload_dataframe(companies,"companies")
upload_dataframe(deals,"funding")

In [1]:
from parser_new_cleaned import keepcool

df = keepcool("https://www.keepcool.co/p/summer-break")


this is the url


In [2]:
df

,name,deal_size,series,tagline,location,investors,domain,date,source
0,RideAlso,$200M,,Small EVs designed for city use and ride-hailing,"Palo Alto, California",Greenoaks Capital,https://ridealso.com/,2025-07-22,https://www.keepcool.co/p/summer-break
1,Climeworks,$162M,,Direct Air Capture technology,"Zurich, Switzerland","BigPoint Holding, Partners Group",https://www.climeworks.com/,2025-07-22,https://www.keepcool.co/p/summer-break
2,Gridserve,$135M,,EV charging stations and solar-powered service...,"Buckinghamshire, U.K.","TPG, Infracapital, Mitsubishi",https://gridserve.com/,2025-07-22,https://www.keepcool.co/p/summer-break
3,Exodigo,$96M,Series B,AI-powered underground infrastructure mapping ...,"Palo Alto, California","Zeev Ventures, Greenfield Partners",https://www.exodigo.com/,2025-07-22,https://www.keepcool.co/p/summer-break
4,Bedrock Robotics,$80M,,Autonomous construction equipment,"San Francisco, California","Eclipse, 8VC",https://bedrockrobotics.com/,2025-07-22,https://www.keepcool.co/p/summer-break
5,Amogy,$80M,,Ammonia-to-power technology for shipping and d...,"Brooklyn, New York","Korea Development Bank, KDB Silicon Valley LLC",https://amogy.co/,2025-07-22,https://www.keepcool.co/p/summer-break
6,Q.ant,$72.5M,Series A,Photonics processors for AI and high-performan...,"Stuttgart, Germany","Cherry Ventures, UVC Partners, imec.xpand",https://qant.com/,2025-07-22,https://www.keepcool.co/p/summer-break
7,Nitricity,$63M,,Organic fertilizer plant using renewable energy,California,"Energy Impact Partners, Khosla Ventures",https://nitricity.co/,2025-07-22,https://www.keepcool.co/p/summer-break
8,BrightAI,$51M,Series A,AI-powered industrial power system management ...,"San Francisco, California","Khosla Ventures, Inspired Capital",https://bright.ai/,2025-07-22,https://www.keepcool.co/p/summer-break
9,GeologicAI,$44M,Series B,Automated rock core analysis for mining companies,"Calgary, Canada","Blue Earth Capital, Rio Tinto, BHP Ventures",https://geologicai.com/,2025-07-22,https://www.keepcool.co/p/summer-break
